# Supervised Fine-Tuning with DeepSpeed

This notebook demonstrates how to use the DeepSpeed trainer for SFT training.

## Initialization

In [ ]:
! uv sync
! git clone https://github.com/ggerganov/llama.cpp

## Configuration

In [7]:
SYSTEM_PROMPT = "邮件类型分析，全部使用中文回答问题。根据邮件内容判断为正常邮件或钓鱼/垃圾邮件。"
base_model_path = "./meta-llama/Llama-3.2-1B-Instruct"
lora_model_path = "./model/lora_model/"
train_data_path = "./data/phish/ourdata1_formatted.csv"
test_data_path = "./data/phish/test_data.csv"

## Prepare dataset

In [ ]:
import csv
from datasets import load_dataset
from tqdm import tqdm


def save_formatted_dataset(formatted_data, filename):
    with open(filename, 'w+', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['line'])
        for row in formatted_data:
            writer.writerow([row])


def formatting_prompts_func(example):
    output_texts = []
    for i in tqdm(range(len(example['eml_content']))):
        USER_PROMPT = example["eml_content"][i][:2800]
        ASSISTANT_PROMPT = f"依据{example["reason"][i]},判断该邮件类型为：{example["type"][i]}。"
        text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>{SYSTEM_PROMPT}<|eot_id|><|start_header_id|>user<|end_header_id|>{USER_PROMPT}<|eot_id|><|start_header_id|>assistant<|end_header_id|>{ASSISTANT_PROMPT}<|eot_id|><|end_of_text|>"""
        output_texts.append(text)
    return output_texts


def load_formatted_dataset(filename):
    with open(filename, 'r') as f:
        reader = csv.reader(f)
        return [row[0] for row in reader]  # Read each row and return as a list


dataset = load_dataset("./data/phish/", data_files=["ourdata1.csv"])
dataset = dataset["train"]
formatted_dataset = formatting_prompts_func(dataset)
save_formatted_dataset(
    formatted_dataset, './data/phish/ourdata1_formatted.csv')

## Training Execution

In [ ]:
! deepspeed --num_gpus=2 train.py

## Test Coverage

In [ ]:
import re
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from peft import PeftModel
import torch
import sys
from datasets import Dataset
from tqdm.auto import tqdm
import json
from datetime import datetime


BATCH_SIZE = 32

# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=False,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
# )

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    # quantization_config=quantization_config,
)

tokenizer = AutoTokenizer.from_pretrained(
    base_model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

new_model = PeftModel.from_pretrained(base_model, lora_model_path)
new_model = new_model.merge_and_unload()

pipe = pipeline(task="text-generation", model=new_model, tokenizer=tokenizer,
                max_new_tokens=200, truncation=False,
                temperature=0.7, clean_up_tokenization_spaces=False,
                batch_size=BATCH_SIZE)


def generate_prompt(text):
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>{SYSTEM_PROMPT}<|eot_id|>
    <|start_header_id|>user<|end_header_id|>{str(text)[:2800]}<|eot_id|>
    <|start_header_id|>assistant<|end_header_id|>
    """


def evaluate_model(outputs, df):
    correct = 0
    total = len(outputs)
    results = []

    for idx, (output, row) in enumerate(zip(outputs, df.iterrows())):
        _, row_data = row
        result_entry = {
            "sample_id": idx + 1,
            "email_content": row_data['line'],
            "is_phishing": bool(row_data['is_phishing']),
            "model_output": output,
            "prediction": "未知"
        }
        if "类型为：正常" in output:
            result_entry["prediction"] = "正常"
        elif "类型为：异常" in output:
            result_entry["prediction"] = "异常"

        if result_entry["prediction"] == "正常" and not bool(row_data['is_phishing']):
            correct += 1
            result_entry["correct"] = True
        elif result_entry["prediction"] == "异常" and bool(row_data['is_phishing']):
            correct += 1
            result_entry["correct"] = True
        else:
            result_entry["correct"] = False

        results.append(result_entry)
        print("="*20+f"Sample {idx+1}"+"="*20)
        print(output)
        print(f"Accuracy: {correct}/{idx+1} {correct/(idx+1):.4f}")

    final_accuracy = correct/total
    print(f"Final Accuracy: {correct}/{total} {final_accuracy:.4f}")
    return total, correct, final_accuracy, results


df = pd.read_csv(test_data_path)
dataset = Dataset.from_pandas(df)

# Prepare prompts
dataset = dataset.map(
    lambda x: {"prompt": generate_prompt(x["line"])}
)

# Generate predictions in batches
outputs = []
for i in tqdm(range(0, len(dataset), pipe._batch_size), desc="Generating predictions"):
    batch = dataset[i:i + pipe._batch_size]["prompt"]
    batch_outputs = pipe(batch)
    # The pipeline returns a list of lists where each inner list contains a dict
    for out in batch_outputs:
        if isinstance(out, list):
            outputs.append(out[0]['generated_text'])
        else:
            outputs.append(out['generated_text'])
        outputs[-1] = re.findall(r"assistant<\|end_header_id\|>.*\s+(.*)",
                                 outputs[-1])[0]
    evaluate_model(outputs, df)

total, correct, final_accuracy, results = evaluate_model(outputs, df)
# Save results to file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"evaluation_results_{timestamp}.json"
output_data = {
    "metadata": {
        "model_checkpoint": sys.argv[1],
        "total_samples": total,
        "correct_predictions": correct,
        "accuracy": final_accuracy,
        "timestamp": timestamp
    },
    "results": results
}

display(output_data)

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"\nResults saved to {output_file}")

## Merge Model

Merge LoRA weights with base model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
from pathlib import Path
import os


def convert_tensor_name(name: str) -> str:
    """Convert HF tensor names to llama.cpp format"""
    name = name.replace("model.layers.", "blk.")
    name = name.replace("self_attn", "attn")
    name = name.replace("q_proj", "wq")
    name = name.replace("k_proj", "wk")
    name = name.replace("v_proj", "wv")
    name = name.replace("o_proj", "wo")
    name = name.replace("gate_proj", "w1")
    name = name.replace("down_proj", "w2")
    name = name.replace("up_proj", "w3")
    name = name.replace("input_layernorm", "norm_1")
    name = name.replace("post_attention_layernorm", "norm_2")
    name = name.replace("norm.weight", "norm.weight")
    name = name.replace("embed_tokens", "token_embd")
    name = name.replace("lm_head", "output")
    return name


def merge_and_convert_to_gguf(
    base_model_path: str = base_model_path,
    lora_model_path: str = lora_model_path,
    output_path: str = "./model/merged_model",
):
    if not os.path.exists(base_model_path):
        raise FileNotFoundError(f"Base model not found at {base_model_path}")
    if not os.path.exists(lora_model_path):
        raise FileNotFoundError(f"LoRA model not found at {lora_model_path}")

    print("Loading base model and tokenizer...")
    try:
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_path,
            low_cpu_mem_usage=True,
            return_dict=True,
            torch_dtype=torch.float16,
            device_map="auto",
        )
        tokenizer = AutoTokenizer.from_pretrained(
            base_model_path,
            trust_remote_code=True
        )
    except Exception as e:
        raise RuntimeError(f"Failed to load base model: {e}")

    print("Loading LoRA model...")
    try:
        model = PeftModel.from_pretrained(base_model, lora_model_path)
    except Exception as e:
        raise RuntimeError(f"Failed to load LoRA model: {e}")

    print("Merging LoRA weights with base model...")
    try:
        merged_model = model.merge_and_unload()
    except Exception as e:
        raise RuntimeError(f"Failed to merge models: {e}")

    # Create output directory if it doesn't exist
    output_dir = Path(output_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    print("Saving merged model...")
    try:
        # Save model with safetensors format
        merged_model.save_pretrained(
            output_path,
            safe_serialization=True,  # Use safetensors format
        )
        tokenizer.save_pretrained(output_path)

        # Verify files exist
        if not os.path.exists(os.path.join(output_path, "model.safetensors")):
            raise RuntimeError("model.safetensors not found after saving")
        if not os.path.exists(os.path.join(output_path, "tokenizer.json")):
            raise RuntimeError("tokenizer.json not found after saving")

    except Exception as e:
        raise RuntimeError(f"Failed to save merged model: {e}")
    return output_path


try:
    merged_model_path = merge_and_convert_to_gguf()
    if merged_model_path:
        print("Conversion completed successfully")
    else:
        print("Conversion failed")
except Exception as e:
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()

## Model Export & Quantization

1. Convert to GGUF format
2. Quantize for efficient inference

In [ ]:
! python3 llama.cpp/convert_hf_to_gguf.py ./model/merged_model --outfile ./model/merged_model/phishkiller_q8_0.gguf --outtype q8_0

## Import to Ollama

In [ ]:
! ollama create phishkiller:1b -f Modelfile

In [ ]:
! ollama show phishkiller:1b